Works but scroll into the end

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

def scroll_spotify_discography():
    # Setup Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    # Remove headless mode so you can see the browser
    # chrome_options.add_argument("--headless")
    
    # Initialize the webdriver
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        # Open the Spotify URL
        url = "https://open.spotify.com/artist/5VVN3xZw1i2qihfITZlvCZ/discography/all"
        print(f"Opening URL: {url}")
        driver.get(url)
        
        # Wait for the page to load
        time.sleep(5)
        
        # Wait for the scrollable content area to be present
        wait = WebDriverWait(driver, 20)
        
        # Find the main content area (right side content) - Updated selectors
        try:
            # Based on the DOM structure visible in the screenshot
            selectors_to_try = [
                # Target the main content container
                '[data-overlayscrollbars-viewport]',
                '.os-viewport',
                # Target elements that have os-scrollbar classes nearby
                'div[class*="os-scrollbar"]',
                # Main content area
                'main[role="main"]',
                'main',
                # Content sections
                'section[data-testid*="discography"]',
                '[data-testid="artist-page"]',
                # General content containers
                'div[style*="overflow"]',
                '[class*="scroll"]'
            ]
            
            viewport = None
            for selector in selectors_to_try:
                try:
                    elements = driver.find_elements(By.CSS_SELECTOR, selector)
                    print(f"Found {len(elements)} elements with selector: {selector}")
                    
                    for i, element in enumerate(elements):
                        try:
                            # Check if element is scrollable and has reasonable size
                            scroll_height = driver.execute_script("return arguments[0].scrollHeight", element)
                            client_height = driver.execute_script("return arguments[0].clientHeight", element)
                            
                            print(f"  Element {i}: ScrollHeight={scroll_height}, ClientHeight={client_height}")
                            
                            if scroll_height > client_height and client_height > 300:
                                viewport = element
                                class_name = element.get_attribute("class") or "no-class"
                                print(f"✓ Found scrollable viewport: {selector}")
                                print(f"  Class: {class_name}")
                                print(f"  Dimensions: {scroll_height}x{client_height}")
                                break
                        except Exception as e:
                            print(f"  Error checking element {i}: {e}")
                            continue
                    
                    if viewport:
                        break
                        
                except Exception as e:
                    print(f"Error with selector {selector}: {e}")
                    continue
            
            # If still not found, try to find the element that contains the discography content
            if not viewport:
                print("Trying to find discography container...")
                try:
                    # Look for elements containing album/song information
                    discography_elements = driver.find_elements(By.XPATH, "//*[contains(text(), 'songs') or contains(text(), 'album') or contains(text(), 'EP')]")
                    for elem in discography_elements:
                        parent = elem.find_element(By.XPATH, "..")
                        while parent:
                            try:
                                scroll_height = driver.execute_script("return arguments[0].scrollHeight", parent)
                                client_height = driver.execute_script("return arguments[0].clientHeight", parent)
                                if scroll_height > client_height and client_height > 300:
                                    viewport = parent
                                    print(f"Found viewport via discography content search")
                                    break
                                parent = parent.find_element(By.XPATH, "..")
                            except:
                                break
                        if viewport:
                            break
                except Exception as e:
                    print(f"Error in discography search: {e}")
            
            if not viewport:
                print("Still couldn't find scrollable content!")
                return
                
        except Exception as e:
            print(f"Error finding viewport: {str(e)}")
            return
        
        # Get initial scroll height
        last_height = driver.execute_script("return arguments[0].scrollHeight", viewport)
        print(f"Initial scroll height: {last_height}")
        
        scroll_count = 0
        max_scrolls = 50  # Safety limit
        
        while scroll_count < max_scrolls:
            # Scroll down within the viewport
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", viewport)
            
            # Wait for new content to load
            time.sleep(2)
            
            # Get new scroll height
            new_height = driver.execute_script("return arguments[0].scrollHeight", viewport)
            
            scroll_count += 1
            print(f"Scroll #{scroll_count}: Height changed from {last_height} to {new_height}")
            
            # Check if we've reached the bottom (no new content loaded)
            if new_height == last_height:
                print("Reached the bottom - no more content to load!")
                break
            
            last_height = new_height
        
        if scroll_count >= max_scrolls:
            print(f"Reached maximum scroll limit ({max_scrolls})")
        
        print("Scrolling completed! Waiting 5 seconds before closing...")
        time.sleep(5)
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")
    
    finally:
        # Close the browser
        driver.quit()
        print("Browser closed.")

def scroll_with_javascript_alternative():
    """Alternative method using pure JavaScript scrolling"""
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        url = "https://open.spotify.com/artist/5VVN3xZw1i2qihfITZlvCZ/discography/all"
        print(f"Opening URL: {url}")
        driver.get(url)
        
        time.sleep(5)
        
        # JavaScript to find and scroll the correct content area - Updated
        scroll_script = """
        // Function to find the scrollable content area based on visible DOM structure
        function findScrollableContent() {
            console.log('Starting search for scrollable content...');
            
            // Priority selectors based on the DOM structure seen
            var prioritySelectors = [
                '[data-overlayscrollbars-viewport]',
                '.os-viewport',
                'main[role="main"]',
                'main'
            ];
            
            // Try priority selectors first
            for (var i = 0; i < prioritySelectors.length; i++) {
                var elements = document.querySelectorAll(prioritySelectors[i]);
                console.log('Found ' + elements.length + ' elements with selector: ' + prioritySelectors[i]);
                
                for (var j = 0; j < elements.length; j++) {
                    var element = elements[j];
                    var scrollHeight = element.scrollHeight;
                    var clientHeight = element.clientHeight;
                    
                    console.log('Element ' + j + ': ScrollHeight=' + scrollHeight + ', ClientHeight=' + clientHeight);
                    
                    if (scrollHeight > clientHeight && clientHeight > 300) {
                        console.log('✓ Found scrollable element with selector: ' + prioritySelectors[i]);
                        console.log('  Classes: ' + (element.className || 'no-class'));
                        return element;
                    }
                }
            }
            
            // Look for elements that have scrollbar-related classes nearby
            var scrollbarElements = document.querySelectorAll('[class*="scrollbar"], [class*="os-scrollbar"]');
            for (var k = 0; k < scrollbarElements.length; k++) {
                var scrollbarEl = scrollbarElements[k];
                var parent = scrollbarEl.parentElement;
                
                while (parent && parent !== document.body) {
                    if (parent.scrollHeight > parent.clientHeight && parent.clientHeight > 300) {
                        console.log('✓ Found scrollable parent of scrollbar element');
                        return parent;
                    }
                    parent = parent.parentElement;
                }
            }
            
            // Look for elements containing discography content
            var textElements = document.querySelectorAll('*');
            for (var l = 0; l < textElements.length; l++) {
                var textEl = textElements[l];
                var text = textEl.textContent || '';
                
                if (text.includes('songs') || text.includes('album') || text.includes('EP') || text.includes('discography')) {
                    var ancestor = textEl.parentElement;
                    var depth = 0;
                    
                    while (ancestor && ancestor !== document.body && depth < 10) {
                        if (ancestor.scrollHeight > ancestor.clientHeight && ancestor.clientHeight > 300) {
                            console.log('✓ Found scrollable ancestor of content element');
                            return ancestor;
                        }
                        ancestor = ancestor.parentElement;
                        depth++;
                    }
                }
            }
            
            console.log('No suitable scrollable element found');
            return null;
        }
        
        var viewport = findScrollableContent();
        
        if (viewport) {
            var lastHeight = viewport.scrollHeight;
            var scrollCount = 0;
            var maxScrolls = 200;
            var scrollIncrement = 300; // Smaller scroll increments
            
            console.log('Starting scroll with element: ' + viewport.tagName);
            console.log('Initial scrollHeight: ' + lastHeight);
            console.log('Initial clientHeight: ' + viewport.clientHeight);
            
            function scrollStep() {
                if (scrollCount < maxScrolls) {
                    // Calculate next scroll position
                    var currentScrollTop = viewport.scrollTop;
                    var maxScrollTop = viewport.scrollHeight - viewport.clientHeight;
                    var nextScrollTop = Math.min(currentScrollTop + scrollIncrement, maxScrollTop);
                    
                    // Perform scroll
                    viewport.scrollTop = nextScrollTop;
                    scrollCount++;
                    
                    console.log('Scroll #' + scrollCount + ': ' + currentScrollTop + ' → ' + nextScrollTop + ' (max: ' + maxScrollTop + ')');
                    
                    setTimeout(function() {
                        var newHeight = viewport.scrollHeight;
                        var currentPosition = viewport.scrollTop;
                        var maxPosition = viewport.scrollHeight - viewport.clientHeight;
                        
                        // Continue if there's new content or we haven't reached the bottom
                        if (newHeight > lastHeight || currentPosition < maxPosition - 50) {
                            lastHeight = newHeight;
                            scrollStep();
                        } else {
                            console.log('✓ Scrolling completed after ' + scrollCount + ' steps');
                            console.log('Final position: ' + currentPosition + '/' + maxPosition);
                            console.log('Final scrollHeight: ' + newHeight);
                        }
                    }, 2000); // Wait 2 seconds between scrolls
                } else {
                    console.log('⚠ Reached maximum scroll limit: ' + maxScrolls);
                }
            }
            
            scrollStep();
            return 'Scrolling started on: ' + viewport.tagName + ' (class: ' + (viewport.className || 'none') + ')';
        } else {
            return 'No scrollable viewport found';
        }
        """
        
        result = driver.execute_script(scroll_script)
        print(f"JavaScript result: {result}")
        
        # Wait for scrolling to complete (increased time)
        print("Waiting for scrolling to complete... (This may take a few minutes)")
        time.sleep(180)  # Wait 3 minutes for scrolling
        
        print("Scrolling should be completed. Check the browser window.")
        input("Press Enter to close the browser...")
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")
    
    finally:
        driver.quit()
        print("Browser closed.")

if __name__ == "__main__":
    print("Spotify Discography Auto-Scroller")
    print("=" * 40)
    
    method = input("Choose method (1 for Python control, 2 for JavaScript control): ")
    
    if method == "2":
        scroll_with_javascript_alternative()
    else:
        scroll_spotify_discography()

Spotify Discography Auto-Scroller
Opening URL: https://open.spotify.com/artist/5VVN3xZw1i2qihfITZlvCZ/discography/all
Found 2 elements with selector: [data-overlayscrollbars-viewport]
  Element 0: ScrollHeight=373, ClientHeight=197
  Element 1: ScrollHeight=296214, ClientHeight=505
✓ Found scrollable viewport: [data-overlayscrollbars-viewport]
  Class: no-class
  Dimensions: 296214x505
Initial scroll height: 296214
Scroll #1: Height changed from 296214 to 296688
Scroll #2: Height changed from 296688 to 296688
Reached the bottom - no more content to load!
Scrolling completed! Waiting 5 seconds before closing...
Browser closed.


In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

def scroll_spotify_discography():
    # Setup Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    # Keep browser visible to see the human-like scrolling
    
    # Initialize the webdriver
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        # Open the Spotify URL
        url = "https://open.spotify.com/artist/5VVN3xZw1i2qihfITZlvCZ/discography/all"
        print(f"Opening URL: {url}")
        driver.get(url)
        
        # Wait for the page to load
        time.sleep(5)
        
        # Wait for the scrollable content area to be present
        wait = WebDriverWait(driver, 20)
        
        # Find the main content area (right side content) - Updated selectors
        try:
            # Based on the DOM structure visible in the screenshot
            selectors_to_try = [
                # Target the main content container
                '[data-overlayscrollbars-viewport]',
                '.os-viewport',
                # Target elements that have os-scrollbar classes nearby
                'div[class*="os-scrollbar"]',
                # Main content area
                'main[role="main"]',
                'main',
                # Content sections
                'section[data-testid*="discography"]',
                '[data-testid="artist-page"]',
                # General content containers
                'div[style*="overflow"]',
                '[class*="scroll"]'
            ]
            
            viewport = None
            for selector in selectors_to_try:
                try:
                    elements = driver.find_elements(By.CSS_SELECTOR, selector)
                    print(f"Found {len(elements)} elements with selector: {selector}")
                    
                    for i, element in enumerate(elements):
                        try:
                            # Check if element is scrollable and has reasonable size
                            scroll_height = driver.execute_script("return arguments[0].scrollHeight", element)
                            client_height = driver.execute_script("return arguments[0].clientHeight", element)
                            
                            print(f"  Element {i}: ScrollHeight={scroll_height}, ClientHeight={client_height}")
                            
                            if scroll_height > client_height and client_height > 300:
                                viewport = element
                                class_name = element.get_attribute("class") or "no-class"
                                print(f"✓ Found scrollable viewport: {selector}")
                                print(f"  Class: {class_name}")
                                print(f"  Dimensions: {scroll_height}x{client_height}")
                                break
                        except Exception as e:
                            print(f"  Error checking element {i}: {e}")
                            continue
                    
                    if viewport:
                        break
                        
                except Exception as e:
                    print(f"Error with selector {selector}: {e}")
                    continue
            
            # If still not found, try to find the element that contains the discography content
            if not viewport:
                print("Trying to find discography container...")
                try:
                    # Look for elements containing album/song information
                    discography_elements = driver.find_elements(By.XPATH, "//*[contains(text(), 'songs') or contains(text(), 'album') or contains(text(), 'EP')]")
                    for elem in discography_elements:
                        parent = elem.find_element(By.XPATH, "..")
                        while parent:
                            try:
                                scroll_height = driver.execute_script("return arguments[0].scrollHeight", parent)
                                client_height = driver.execute_script("return arguments[0].clientHeight", parent)
                                if scroll_height > client_height and client_height > 300:
                                    viewport = parent
                                    print(f"Found viewport via discography content search")
                                    break
                                parent = parent.find_element(By.XPATH, "..")
                            except:
                                break
                        if viewport:
                            break
                except Exception as e:
                    print(f"Error in discography search: {e}")
            
            if not viewport:
                print("Still couldn't find scrollable content!")
                return
                
        except Exception as e:
            print(f"Error finding viewport: {str(e)}")
            return
        
        # Get initial scroll position and height
        initial_scroll_top = driver.execute_script("return arguments[0].scrollTop", viewport)
        initial_scroll_height = driver.execute_script("return arguments[0].scrollHeight", viewport)
        client_height = driver.execute_script("return arguments[0].clientHeight", viewport)
        
        print(f"Initial scroll position: {initial_scroll_top}")
        print(f"Initial scroll height: {initial_scroll_height}")
        print(f"Client height: {client_height}")
        
        # Continuous scrolling parameters
        scroll_step = 150  # Small scroll steps for smooth scrolling
        scroll_delay = 0.3  # Fast continuous scrolling (300ms between scrolls)
        
        scroll_count = 0
        max_scrolls = 1000  # Higher safety limit
        last_height = initial_scroll_height
        no_new_content_count = 0
        
        print("Starting continuous scrolling...")
        
        while scroll_count < max_scrolls:
            # Get current scroll position
            current_scroll_top = driver.execute_script("return arguments[0].scrollTop", viewport)
            current_height = driver.execute_script("return arguments[0].scrollHeight", viewport)
            max_scroll_top = current_height - client_height
            
            # Check if we've reached the bottom
            if current_scroll_top >= max_scroll_top - 20:  # 20px tolerance
                if current_height == last_height:
                    no_new_content_count += 1
                    if no_new_content_count >= 5:  # No new content for 5 checks
                        print("✓ Reached the bottom - no more content!")
                        break
                else:
                    no_new_content_count = 0  # Reset counter if new content appeared
                    last_height = current_height
            
            # Calculate next scroll position
            next_scroll_top = min(current_scroll_top + scroll_step, max_scroll_top)
            
            # Perform the scroll
            driver.execute_script("arguments[0].scrollTop = arguments[1]", viewport, next_scroll_top)
            
            scroll_count += 1
            if scroll_count % 20 == 0:  # Show progress every 20 scrolls to avoid spam
                print(f"Scroll #{scroll_count}: Position {next_scroll_top}px (Height: {current_height}px)")
            
            # Fast continuous scrolling
            time.sleep(scroll_delay)
        
        if scroll_count >= max_scrolls:
            print(f"⚠ Reached maximum scroll limit ({max_scrolls})")
        
        # Final position info
        final_scroll_top = driver.execute_script("return arguments[0].scrollTop", viewport)
        final_height = driver.execute_script("return arguments[0].scrollHeight", viewport)
        print(f"\n📊 Scrolling Summary:")
        print(f"   Total scrolls: {scroll_count}")
        print(f"   Initial height: {initial_scroll_height}px")
        print(f"   Final height: {final_height}px")
        print(f"   Final position: {final_scroll_top}px")
        print(f"   Content loaded: {final_height - initial_scroll_height}px")
        
        print("\n✅ Continuous scrolling completed!")
        print("Keeping browser open for 5 seconds...")
        time.sleep(5)
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")
    
    finally:
        # Close the browser
        driver.quit()
        print("Browser closed.")

if __name__ == "__main__":
    print("🎵 Spotify Discography Human-Like Auto-Scroller")
    print("=" * 50)
    scroll_spotify_discography()

🎵 Spotify Discography Human-Like Auto-Scroller
Opening URL: https://open.spotify.com/artist/5VVN3xZw1i2qihfITZlvCZ/discography/all
Found 2 elements with selector: [data-overlayscrollbars-viewport]
  Element 0: ScrollHeight=373, ClientHeight=197
  Element 1: ScrollHeight=296214, ClientHeight=505
✓ Found scrollable viewport: [data-overlayscrollbars-viewport]
  Class: no-class
  Dimensions: 296214x505
Initial scroll position: 0
Initial scroll height: 296214
Client height: 505
Starting continuous scrolling...
Scroll #20: Position 3007.60009765625px (Height: 296917px)
Scroll #40: Position 6015.60009765625px (Height: 295497px)
Scroll #60: Position 9023.599609375px (Height: 295077px)
Scroll #80: Position 12031.599609375px (Height: 296686px)
Scroll #100: Position 15039.599609375px (Height: 295913px)
Scroll #120: Position 18047.599609375px (Height: 295595px)
Scroll #140: Position 21055.599609375px (Height: 295459px)
Scroll #160: Position 24063.599609375px (Height: 294167px)
Scroll #180: Positio